In [3]:
!nvidia-smi

Mon Nov 23 08:54:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
%%capture
!pip install unsloth
!pip install datasets

In [5]:
from datasets import load_dataset

dataset = load_dataset("b-mc2/sql-create-context", split="train")
print(len(dataset))
print(dataset[0])

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

78577
{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}


In [6]:
dataset = dataset.shuffle(seed=42)

test_data = dataset.select(range(200))
train_data = dataset.select(range(200, 5200))

print(len(train_data), len(test_data))

5000 200


In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=1024,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1123 08:55:28.265000 458 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1123 08:55:28.341000 458 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen

In [8]:
def build_prompt(question, context):
    messages = [
        {"role": "system", "content": "You are a SQL expert. Given a table schema and a question, write one SQL query. Output only the SQL."},
        {"role": "user", "content": f"Schema:\n{context}\n\nQuestion: {question}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

example = test_data[0]
prompt = build_prompt(example["question"], example["context"])
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=128, do_sample=False)
generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Question:", example["question"])
print("Expected:", example["answer"])
print("Model:   ", generated)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: When Essendon played away; where did they play?
Expected: SELECT venue FROM table_name_50 WHERE away_team = "essendon"
Model:    SELECT venue FROM table_name_50 WHERE away_team = 'Essendon'


In [9]:
import re
from tqdm import tqdm

def normalize_sql(sql):
    sql = re.sub(r"```(?:sql)?", "", sql)
    sql = sql.strip().rstrip(";").lower()
    sql = sql.replace('"', "'")
    sql = re.sub(r"\s+", " ", sql)
    return sql

def generate_sql(example):
    prompt = build_prompt(example["question"], example["context"])
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluate(data):
    correct, results = 0, []
    for ex in tqdm(data):
        pred = generate_sql(ex)
        match = normalize_sql(pred) == normalize_sql(ex["answer"])
        correct += match
        results.append({"question": ex["question"], "expected": ex["answer"], "predicted": pred, "match": match})
    return correct / len(data), results

base_acc, base_results = evaluate(test_data)
print(f"Base model accuracy: {base_acc:.2%}")

100%|██████████| 200/200 [02:04<00:00,  1.61it/s]

Base model accuracy: 49.00%


In [10]:
import json

with open("base_results.json", "w") as f:
    json.dump({"accuracy": base_acc, "results": base_results}, f, indent=2)

from google.colab import files
files.download("base_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
!pip install sentence-transformers faiss-cpu -q

In [12]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

train_texts = [f"{ex['question']} | {ex['context']}" for ex in train_data]
train_emb = embedder.encode(train_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(train_emb.shape[1])
index.add(np.array(train_emb, dtype="float32"))
print(index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

5000


In [13]:
def retrieve(example, k=3):
    query = f"{example['question']} | {example['context']}"
    q_emb = embedder.encode([query], normalize_embeddings=True)
    _, ids = index.search(np.array(q_emb, dtype="float32"), k)
    return [train_data[int(i)] for i in ids[0]]

def build_rag_prompt(example, k=3):
    messages = [{"role": "system", "content": "You are a SQL expert. Given a table schema and a question, write one SQL query. Output only the SQL."}]
    for shot in retrieve(example, k):
        messages.append({"role": "user", "content": f"Schema:\n{shot['context']}\n\nQuestion: {shot['question']}"})
        messages.append({"role": "assistant", "content": shot["answer"]})
    messages.append({"role": "user", "content": f"Schema:\n{example['context']}\n\nQuestion: {example['question']}"})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

example = test_data[0]
prompt = build_rag_prompt(example)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=128, do_sample=False)
generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Expected:", example["answer"])
print("RAG:     ", generated)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Expected: SELECT venue FROM table_name_50 WHERE away_team = "essendon"
RAG:      SELECT venue FROM table_name_50 WHERE away_team = 'Essendon'


In [14]:
def generate_rag_sql(example):
    prompt = build_rag_prompt(example)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluate_rag(data):
    correct, results = 0, []
    for ex in tqdm(data):
        pred = generate_rag_sql(ex)
        match = normalize_sql(pred) == normalize_sql(ex["answer"])
        correct += match
        results.append({"question": ex["question"], "expected": ex["answer"], "predicted": pred, "match": match})
    return correct / len(data), results

rag_acc, rag_results = evaluate_rag(test_data)
print(f"RAG accuracy: {rag_acc:.2%}")

with open("rag_results.json", "w") as f:
    json.dump({"accuracy": rag_acc, "results": rag_results}, f, indent=2)
files.download("rag_results.json")

100%|██████████| 200/200 [02:47<00:00,  1.19it/s]

RAG accuracy: 56.00%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
def format_example(ex):
    messages = [
        {"role": "system", "content": "You are a SQL expert. Given a table schema and a question, write one SQL query. Output only the SQL."},
        {"role": "user", "content": f"Schema:\n{ex['context']}\n\nQuestion: {ex['question']}"},
        {"role": "assistant", "content": ex["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_formatted = train_data.map(format_example, remove_columns=train_data.column_names)

print(len(train_formatted))
print(train_formatted[0]["text"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

5000
<|im_start|>system
You are a SQL expert. Given a table schema and a question, write one SQL query. Output only the SQL.<|im_end|>
<|im_start|>user
Schema:
CREATE TABLE table_name_69 (height VARCHAR, name VARCHAR)

Question: How tall is Dylan Postl?<|im_end|>
<|im_start|>assistant
SELECT height FROM table_name_69 WHERE name = "dylan postl"<|im_end|>



In [16]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth 2026.11.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [17]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth: Already have LoRA adapters! We shall skip this step.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [18]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=1024,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_steps=10,
        logging_steps=10,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        output_dir="outputs",
        seed=42,
        report_to="none",
    ),
)
print("Trainer ready")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: Packing is on by default - documents are packed into full rows with no cross-document attention. Set packing = False in SFTConfig (or UNSLOTH_DEFAULT_PACKING=0) to turn it off.
Trainer ready


In [19]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 518 | Num Epochs = 1 | Total steps = 33
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.506909
20,0.910390
30,0.674781


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-33/tokenizer_config.json.


In [20]:
model.save_pretrained("sql-lora-adapter")
tokenizer.save_pretrained("sql-lora-adapter")

!zip -r sql-lora-adapter.zip sql-lora-adapter
files.download("sql-lora-adapter.zip")

Unsloth: Restored added_tokens_decoder metadata in sql-lora-adapter/tokenizer_config.json.


  adding: sql-lora-adapter/ (stored 0%)
  adding: sql-lora-adapter/tokenizer.json (deflated 81%)
  adding: sql-lora-adapter/README.md (deflated 65%)
  adding: sql-lora-adapter/adapter_config.json (deflated 60%)
  adding: sql-lora-adapter/adapter_model.safetensors (deflated 8%)
  adding: sql-lora-adapter/chat_template.jinja (deflated 71%)
  adding: sql-lora-adapter/tokenizer_config.json (deflated 89%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
FastLanguageModel.for_inference(model)

ft_acc, ft_results = evaluate(test_data)
print(f"Fine-tuned accuracy: {ft_acc:.2%}")

with open("ft_results.json", "w") as f:
    json.dump({"accuracy": ft_acc, "results": ft_results}, f, indent=2)
files.download("ft_results.json")

100%|██████████| 200/200 [02:31<00:00,  1.32it/s]

Fine-tuned accuracy: 65.50%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
ft_rag_acc, ft_rag_results = evaluate_rag(test_data)
print(f"Fine-tuned + RAG accuracy: {ft_rag_acc:.2%}")

with open("ft_rag_results.json", "w") as f:
    json.dump({"accuracy": ft_rag_acc, "results": ft_rag_results}, f, indent=2)
files.download("ft_rag_results.json")


100%|██████████| 200/200 [02:43<00:00,  1.22it/s]

Fine-tuned + RAG accuracy: 68.00%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:

failures = [r for r in ft_results if not r["match"]]
print(f"{len(failures)} failures out of {len(ft_results)}\n")

for r in failures[:15]:
    print("Q:       ", r["question"])
    print("Expected:", r["expected"])
    print("Got:     ", r["predicted"])
    print("-" * 60)

69 failures out of 200

Q:        Which Points have Touchdowns larger than 0, and an Extra points smaller than 0?
Expected: SELECT SUM(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0
Got:      SELECT AVG(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0
------------------------------------------------------------
Q:        What is the name of the player who is Sco and moving to greenock morton in the summer?
Expected: SELECT name FROM table_name_83 WHERE nat = "sco" AND transfer_window = "summer" AND moving_to = "greenock morton"
Got:      SELECT name FROM table_name_83 WHERE moving_to = "greenock morton" AND transfer_window = "summer" AND nat = "SCO"
------------------------------------------------------------
Q:        Of all the contestants who got voted, what is the contestant number and name of the contestant who got least votes?
Expected: SELECT T1.contestant_number, T1.contestant_name FROM contestants AS T1 JOIN votes AS T2 ON T1.contestan

In [24]:
def normalize_v2(sql):
    s = normalize_sql(sql)
    s = re.sub(r"'(\d+(?:\.\d+)?)'", r"\1", s)  # '1990' -> 1990
    if " where " in s:
        head, where = s.split(" where ", 1)
        if not any(t in where for t in [" or ", "between", "(", " group by ", " order by ", " limit "]):
            conds = sorted(c.strip() for c in where.split(" and "))
            s = head + " where " + " and ".join(conds)
    return s

for name, res in [("Base", base_results), ("Base + RAG", rag_results),
                  ("Fine-tuned", ft_results), ("Fine-tuned + RAG", ft_rag_results)]:
    strict = sum(r["match"] for r in res) / len(res)
    relaxed = sum(normalize_v2(r["predicted"]) == normalize_v2(r["expected"]) for r in res) / len(res)
    print(f"{name:18} strict: {strict:.1%}   relaxed: {relaxed:.1%}")

Base               strict: 49.0%   relaxed: 62.0%
Base + RAG         strict: 56.0%   relaxed: 66.5%
Fine-tuned         strict: 65.5%   relaxed: 75.5%
Fine-tuned + RAG   strict: 68.0%   relaxed: 77.0%


In [26]:
from huggingface_hub import notebook_login
notebook_login()

In [27]:
from huggingface_hub import HfApi

username = HfApi().whoami()["name"]
repo_id = f"{username}/qwen2.5-3b-text2sql-qlora"

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"https://huggingface.co/{repo_id}")

README.md:   0%|          | 0.00/569 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  562kB /  120MB            

Validating                    :           |   0%            

Saved model to https://huggingface.co/Nive45/qwen2.5-3b-text2sql-qlora


... (2 outputs left out)

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp1clzvr1t/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

https://huggingface.co/Nive45/qwen2.5-3b-text2sql-qlora


In [28]:
!pip install gradio -q
import gradio as gr

def ask(question, schema):
    return generate_sql({"question": question, "context": schema})

demo = gr.Interface(
    fn=ask,
    inputs=[
        gr.Textbox(label="Question"),
        gr.Textbox(label="Table schema (CREATE TABLE ...)", lines=4),
    ],
    outputs=gr.Textbox(label="Generated SQL"),
    title="Text-to-SQL (Qwen2.5-3B + QLoRA)",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://95b78d8add37f3bdca.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:
tests = [
    ("Names of employees in sales earning over 50000",
     "CREATE TABLE employees (id INT, name VARCHAR, salary INT, dept VARCHAR)"),
    ("How many students are older than 20?",
     "CREATE TABLE students (student_id INT, full_name VARCHAR, age INT, major VARCHAR)"),
    ("List the titles of books published after 2010, newest first",
     "CREATE TABLE books (book_id INT, title VARCHAR, author VARCHAR, year_published INT)"),
]

for q, s in tests:
    print("Q:", q)
    print("Schema:", s)
    print("SQL:", generate_sql({"question": q, "context": s}))
    print("-" * 50)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Names of employees in sales earning over 50000
Schema: CREATE TABLE employees (id INT, name VARCHAR, salary INT, dept VARCHAR)


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL: SELECT name FROM employees WHERE salary > 50000 AND dept = "sales"
--------------------------------------------------
Q: How many students are older than 20?
Schema: CREATE TABLE students (student_id INT, full_name VARCHAR, age INT, major VARCHAR)


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL: SELECT COUNT(*) FROM students WHERE age > 20
--------------------------------------------------
Q: List the titles of books published after 2010, newest first
Schema: CREATE TABLE books (book_id INT, title VARCHAR, author VARCHAR, year_published INT)
SQL: SELECT title FROM books WHERE year_published > 2010 ORDER BY year_published DESC
--------------------------------------------------


In [31]:
demo.close()

def ask(question, schema):
    sql = generate_sql({"question": question, "context": schema})
    return f"{sql}\n\n[debug] question={question!r}\n[debug] schema={schema!r}"

demo = gr.Interface(
    fn=ask,
    inputs=[
        gr.Textbox(label="Question"),
        gr.Textbox(label="Table schema (CREATE TABLE ...)", lines=4),
    ],
    outputs=gr.Textbox(label="Generated SQL", lines=6),
    title="Text-to-SQL (debug)",
)
demo.launch(share=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7793a761fe0ebece03.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [32]:
demo.close()

def ask(question, schema):
    return generate_sql({"question": question, "context": schema})

demo = gr.Interface(
    fn=ask,
    inputs=[
        gr.Textbox(label="Question"),
        gr.Textbox(label="Table schema (CREATE TABLE ...)", lines=4),
    ],
    outputs=gr.Textbox(label="Generated SQL"),
    title="Text-to-SQL (Qwen2.5-3B + QLoRA)",
)
demo.launch(share=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5af34f5f77e5155402.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
